In [24]:
%load_ext autoreload
%autoreload 2
import argparse
import json
from typing import Any, cast

from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.state import CompiledStateGraph
from rich import print
from support_agent.config import Settings, configure_langsmith
from support_agent.graph import build_support_graph
from support_agent.knowledge_base import HybridChromaKnowledgeBase
from support_agent.logging_utils import setup_logger
from support_agent.state import SupportTicketState
from support_agent.test_cases import TEST_CASES_50
from support_agent.utils import format_messages

settings = Settings.from_env()
kb = HybridChromaKnowledgeBase.from_markdown_files(settings=settings)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:

configure_langsmith(settings)
logger = setup_logger(settings.log_level, log_file = f"logs/support_agent_1.log")
checkpointer = MemorySaver()

app = build_support_graph(
    
    settings=settings,
    kb=kb,
    checkpointer=checkpointer,
)

In [31]:
user_text = "Какие метрики относятся к Мультиагентным системам?"
threshold = 0.5
max_model_retries = 1
state: dict[str, Any] = {
        "ticket_id": "1",
        "user_text": user_text,
        "messages": [HumanMessage(content=user_text)],
        "quality_threshold": threshold if threshold is not None else settings.quality_threshold,
        "max_model_retries": (
            max_model_retries if max_model_retries is not None else settings.max_model_retries
        ),
    }
config = RunnableConfig(configurable={"thread_id": "4"})
# for chunk in app.stream(state, config=config):
#     print(chunk)


In [32]:
config = RunnableConfig(configurable={"thread_id": "1"})
message = {"messages": [HumanMessage(content="Какие метрики относятся к Мультиагентным системам?")]}
for chunk in app.stream(state, config=config):
    format_messages(chunk)

2026-05-28 21:42:06 | INFO | support_agent.graph | ReceiveRequest node started for ticket=1


╭───────────────────────────────────────────── Node: ReceiveRequest ──────────────────────────────────────────────╮
│ event: request_received                                                                                         │
│ timestamp: 2026-05-28T18:42:06.532679+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "ticket_id": "1"                                                                                              │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 21:42:06 | INFO | support_agent.graph | ClassifyUrgency node started for ticket=1
2026-05-28 21:42:06 | INFO | support_agent.graph | ClassifyComplaint node started for ticket=1
2026-05-28 21:42:06 | INFO | support_agent.graph | ClassifyCategory node started for ticket=1
2026-05-28 21:42:07 | INFO | support_agent.graph | ClassifyCategory node completed for ticket=1 with category=technical_question
2026-05-28 21:42:07 | INFO | support_agent.graph | ClassifyComplaint node completed for ticket=1 with is_complaint=False


╭──────────────────────────────────────────── Node: ClassifyCategory ─────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T18:42:07.968710+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "category": "technical_question"                                                                              │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Node: ClassifyComplaint ────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T18:42:07.969266+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "is_complaint": false                                                                                         │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Качество ответа ниже порога. Проблемы: Ответ не содержит никакой полезной информации.. Перегенерируй ответ:     │
│ исправь полноту, вежливость и релевантность; используй knowledge_base_search, если нужны факты; если данных     │
│ нет, ответь ровно 'Не знаю'.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🤖 Assistant ──────────────────────────────────────────────────╮
│ 🔧 Tool Call: knowledge_base_search Args: { "query": "Мультиагентные системы метрики" } ID:                     │
│ 9cbefef4-c99c-43ad-8992-beccbb9fd9af                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ База знаний вернула: ['\n[# Фрагмент 0 | source=dataset/kb_doc_3.md]\nАктуальный статус доступен на странице    │
│ «Статус сервиса» в футере личного кабинета.', '\n[# Фрагмент 1 | source=dataset/kb_doc_2.md]\nЛимиты и текущий  │
│ расход доступны в «Разработчикам -> API -> Квоты».', '\n[# Фрагмент 2 | source=dataset/kb_doc_1.md]\nВ          │
│ «Настройки -> Оплата» выберите карту и нажмите «Удалить». Убедитесь, что есть другой активный способ оплаты.']  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🤖 Assistant ──────────────────────────────────────────────────╮
│ Не знаю                                                                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 21:42:17 | INFO | support_agent.graph | FinalizeResponse node started for ticket=1
2026-05-28 21:42:17 | INFO | support_agent.graph | FinalizeResponse node completed for ticket=1 (response_len=7)


In [ ]:
from rich import print

print(app.get_state(config).values)

In [39]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig

# можно запустить с разными настройками, чтобы проверить разные ветки логики
def run_case(text, state_extra=None, thread_id="manual-test"):
    state = {
        "ticket_id": 1,
        "user_text": text,
        "messages": [HumanMessage(content=text)],
        "quality_threshold": 0.9,   # для проверки retry удобно высокий порог
        "max_model_retries": 1,
    }
    if state_extra:
        state.update(state_extra)

    config = RunnableConfig(configurable={"thread_id": thread_id})
    chunks = []
    for chunk in app.stream(state, config=config):
        format_messages(chunk)
        chunks.append(chunk)
    return chunks

In [ ]:
run_case("Как включить темную тему?Срочно! умираю!", thread_id="prio-1")


# run_case("???", state_extra={"quality_threshold": 0.99, "max_model_retries": 1}, thread_id="retry-1")

In [ ]:
# Пояснения к кейсу:

# в обновлениях DialogAgent должен появиться priority_instruction из-за срочности
# 
# в events должно быть событие priority_instruction_applied

run_case("Как включить темную тему?Срочно! умираю!", thread_id="prio-1")

2026-05-28 22:03:59 | INFO | support_agent.graph | ReceiveRequest node started for ticket=1


╭───────────────────────────────────────────── Node: ReceiveRequest ──────────────────────────────────────────────╮
│ event: request_received                                                                                         │
│ timestamp: 2026-05-28T19:03:59.390081+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "ticket_id": 1                                                                                                │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 22:03:59 | INFO | support_agent.graph | ClassifyUrgency node started for ticket=1
2026-05-28 22:03:59 | INFO | support_agent.graph | ClassifyCategory node started for ticket=1
2026-05-28 22:03:59 | INFO | support_agent.graph | ClassifyComplaint node started for ticket=1
2026-05-28 22:04:02 | INFO | support_agent.graph | ClassifyUrgency node completed for ticket=1 with urgency=high
2026-05-28 22:04:02 | INFO | support_agent.graph | ClassifyCategory node completed for ticket=1 with category=how_to
2026-05-28 22:04:02 | INFO | support_agent.graph | ClassifyComplaint node completed for ticket=1 with is_complaint=False


╭───────────────────────────────────────────── Node: ClassifyUrgency ─────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T19:04:02.443934+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "urgency": "high"                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Node: ClassifyCategory ─────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T19:04:02.444358+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "category": "how_to"                                                                                          │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Как включить темную тему?Срочно! умираю!                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🤖 Assistant ──────────────────────────────────────────────────╮
│ 🔧 Tool Call: knowledge_base_search Args: { "query": "Как включить темную тему" } ID:                           │
│ 40bb0ad3-fa34-4463-b5a8-3fdc96e94f73                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ База знаний вернула: ['\n[# Фрагмент 0 | source=dataset/kb_doc_3.md]\nПерейдите в «Профиль -> Оформление» и     │
│ выберите режим «Темная тема».', '\n[# Фрагмент 1 | source=dataset/kb_doc_2.md]\nВ «Настройки -> Уведомления»    │
│ отключите лишние категории и оставьте включенными только критичные события.', '\n[# Фрагмент 2 |                │
│ source=dataset/kb_doc_1.md]\nВ «Настройки -> Оплата» выберите карту и нажмите «Удалить». Убедитесь, что есть    │
│ другой активный способ оплаты.']                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🤖 Assistant ──────────────────────────────────────────────────╮
│ Пожалуйста, перейдите в раздел "Профиль", затем во вкладку "Оформление" и выберите опцию "Темная тема".         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 22:04:08 | INFO | support_agent.graph | FinalizeResponse node started for ticket=1
2026-05-28 22:04:08 | INFO | support_agent.graph | FinalizeResponse node completed for ticket=1 (response_len=103)


In [ ]:
# Пояснения к кейсу:

# priority_instruction отсутствует
# события priority_instruction_applied нет
# quality_retry_requested - сигнал о том, что модель не смогла выполнить задачу из-за высокого порога качества и запросила повтор
# "retry_count": 1 - показывает, что была предпринята попытка повторного выполнения для улучшения качества ответа
run_case("Привет", state_extra={"urgency": "low"}, thread_id="prio-2")

2026-05-28 22:05:13 | INFO | support_agent.graph | ReceiveRequest node started for ticket=1


╭───────────────────────────────────────────── Node: ReceiveRequest ──────────────────────────────────────────────╮
│ event: request_received                                                                                         │
│ timestamp: 2026-05-28T19:05:13.940585+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "ticket_id": 1                                                                                                │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 22:05:13 | INFO | support_agent.graph | ClassifyUrgency node started for ticket=1
2026-05-28 22:05:13 | INFO | support_agent.graph | ClassifyCategory node started for ticket=1
2026-05-28 22:05:13 | INFO | support_agent.graph | ClassifyComplaint node started for ticket=1
2026-05-28 22:05:15 | INFO | support_agent.graph | ClassifyComplaint node completed for ticket=1 with is_complaint=False
2026-05-28 22:05:16 | INFO | support_agent.graph | ClassifyUrgency node completed for ticket=1 with urgency=low


╭──────────────────────────────────────────── Node: ClassifyComplaint ────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T19:05:15.764078+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "is_complaint": false                                                                                         │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Node: ClassifyUrgency ─────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T19:05:16.487588+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "urgency": "low"                                                                                              │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Привет                                                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🤖 Assistant ──────────────────────────────────────────────────╮
│ По этой теме я не поддерживаю диалог.                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Качество ответа ниже порога. Проблемы: Ответ не отвечает на вопрос и не предоставляет полезной информации..     │
│ Перегенерируй ответ: исправь полноту, вежливость и релевантность; используй knowledge_base_search, если нужны   │
│ факты; если данных нет, ответь ровно 'Не знаю'.                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────── 🤖 Assistant ──────────────────────────────────────────────────╮
│ Здравствуйте! Чем конкретно могу вам помочь?                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 22:05:26 | INFO | support_agent.graph | FinalizeResponse node started for ticket=1
2026-05-28 22:05:26 | INFO | support_agent.graph | FinalizeResponse node completed for ticket=1 (response_len=44)


In [40]:
# Пояснения к кейсу:

# классификация обращения как жалобы
# передача в ноду EscalateComplaint 

final_message = run_case("Мне все надоело, ничего не работает!!! (((", thread_id="complaint-1")

2026-05-28 22:14:15 | INFO | support_agent.graph | ReceiveRequest node started for ticket=1


╭───────────────────────────────────────────── Node: ReceiveRequest ──────────────────────────────────────────────╮
│ event: request_received                                                                                         │
│ timestamp: 2026-05-28T19:14:15.641582+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "ticket_id": 1                                                                                                │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 22:14:15 | INFO | support_agent.graph | ClassifyUrgency node started for ticket=1
2026-05-28 22:14:15 | INFO | support_agent.graph | ClassifyCategory node started for ticket=1
2026-05-28 22:14:15 | INFO | support_agent.graph | ClassifyComplaint node started for ticket=1
2026-05-28 22:14:17 | INFO | support_agent.graph | ClassifyUrgency node completed for ticket=1 with urgency=high
2026-05-28 22:14:17 | INFO | support_agent.graph | ClassifyCategory node completed for ticket=1 with category=technical_question
2026-05-28 22:14:17 | INFO | support_agent.graph | ClassifyComplaint node completed for ticket=1 with is_complaint=True


╭───────────────────────────────────────────── Node: ClassifyUrgency ─────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T19:14:17.173848+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "urgency": "high"                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Node: ClassifyCategory ─────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T19:14:17.174185+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "category": "technical_question"                                                                              │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Node: ClassifyComplaint ────────────────────────────────────────────╮
│ event: classification_done                                                                                      │
│ timestamp: 2026-05-28T19:14:17.175711+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "is_complaint": true                                                                                          │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 22:14:17 | INFO | support_agent.graph | GatherClassifications node started for ticket=1 (is_complaint=True, category=technical_question, urgency=high)


╭────────────────────────────────────────── Node: GatherClassifications ──────────────────────────────────────────╮
│ event: gather_completed                                                                                         │
│ timestamp: 2026-05-28T19:14:17.178719+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "is_complaint": true,                                                                                         │
│   "category": "technical_question",                                                                             │
│   "urgency": "high"                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-05-28 22:14:17 | INFO | support_agent.graph | EscalateComplaint node started for ticket=1 (category=technical_question, urgency=high)
2026-05-28 22:14:17 | INFO | support_agent.graph | EscalateComplaint node completed for ticket=1


╭──────────────────────────────────────────── Node: EscalateComplaint ────────────────────────────────────────────╮
│ event: escalated                                                                                                │
│ timestamp: 2026-05-28T19:14:17.180612+00:00                                                                     │
│ payload:                                                                                                        │
│ {                                                                                                               │
│   "reason": "{\"reason\": \"complaint_detected\", \"user_query\": \"Мне все надоело, ничего не работает!!!      │
│ (((\", \"category\": \"technical_question\", \"urgency\": \"high\"}"                                            │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [42]:
print(final_message[-1])

{
    'EscalateComplaint': {
        'escalated': True,
        'escalation_reason': '{"reason": "complaint_detected", "user_query": "Мне все надоело, ничего не 
работает!!! (((", "category": "technical_question", "urgency": "high"}',
        'final_response': 'Ваше обращение передано оператору. Специалист свяжется с вами для детального 
рассмотрения ситуации.',
        'events': [
            {
                'timestamp': '2026-05-28T19:14:17.180612+00:00',
                'ticket_id': 1,
                'node': 'EscalateComplaint',
                'event': 'escalated',
                'payload': {
                    'reason': '{"reason": "complaint_detected", "user_query": "Мне все надоело, ничего не 
работает!!! (((", "category": "technical_question", "urgency": "high"}'
                }
            }
        ]
    }
}